# COCOS Conversion, Bt/Ip Flip, and Save

Demonstrates how to:
1. Load a g-file and inspect its COCOS convention
2. Convert between COCOS conventions (`cocosify`)
3. Flip Bt/Ip directions (`flip_Bt_Ip`)
4. Save the modified equilibrium to disk or HDF5

This replicates the OMFIT workflow:
```python
new_eq = OMFITgeqdsk(eqfile)
new_eq.load(raw=True)
new_eq._cocos = 7
new_eq.cocosify(1, True, True)
new_eq.flip_Bt_Ip()
```

In [3]:
import sys, os, tempfile
import numpy as np

# Ensure bouquet is importable from the repo root
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

from bouquet.io.geqdsk import GEQDSKEquilibrium, read_geqdsk

## 1. Load and inspect the DIII-D-like equilibrium

In [4]:
gfile_path = '../D3D-like/TkMkr_D3Dlike_Hmode_BSamp=1.0.geqdsk'
eq = read_geqdsk(gfile_path, cocos=1)

print(f'COCOS:   {eq.cocos}')
print(f'Ip:      {eq.Ip:,.0f} A')
print(f'B0:      {eq.B_center:.3f} T')
print(f'psi_ax:  {eq.psi_axis:.6f} Wb/rad')
print(f'psi_bdy: {eq.psi_boundary:.6f} Wb/rad')
print(f'FPOL[0]: {eq.fpol[0]:.4f} T·m')
print(f'q(0):    {eq.qpsi[0]:.4f}')

COCOS:   1
Ip:      1,200,040 A
B0:      2.024 T
psi_ax:  0.348276 Wb/rad
psi_bdy: 0.044025 Wb/rad
FPOL[0]: 3.4923 T·m
q(0):    1.2095


## 2. Convert COCOS: 1 → 7 → 1 (round-trip)

COCOS 7 differs from COCOS 1 in the sign of `sigma_Bp` (psi decreases outward)
and `sigma_rhotp`.  The conversion flips the sign of psi fields and q.

In [5]:
# Convert to COCOS 7 (copy=True → original unchanged)
eq7 = eq.cocosify(7, copy=True)

print('--- After cocosify(7) ---')
print(f'COCOS:   {eq7.cocos}')
print(f'Ip:      {eq7.Ip:,.0f} A   (unchanged — same sigma_RpZ)')
print(f'B0:      {eq7.B_center:.3f} T   (unchanged)')
print(f'psi_bdy: {eq7.psi_boundary:.6f}  (sign flipped)')
print(f'q(0):    {eq7.qpsi[0]:.4f}      (sign flipped)')

--- After cocosify(7) ---
COCOS:   7
Ip:      1,200,040 A   (unchanged — same sigma_RpZ)
B0:      2.024 T   (unchanged)
psi_bdy: -0.044025  (sign flipped)
q(0):    1.2095      (sign flipped)


In [6]:
# Round-trip back — should recover original
eq_back = eq7.cocosify(1, copy=True)

print('--- Round-trip back to COCOS 1 ---')
print(f'psi_bdy matches: {np.isclose(eq_back.psi_boundary, eq.psi_boundary)}')
print(f'q(0) matches:    {np.isclose(eq_back.qpsi[0], eq.qpsi[0])}')
print(f'psi_RZ matches:  {np.allclose(eq_back.psi_RZ, eq.psi_RZ)}')

--- Round-trip back to COCOS 1 ---
psi_bdy matches: True
q(0) matches:    True
psi_RZ matches:  True


## 3. Convert COCOS 1 → 11 (the 2π factor)

COCOS 11 uses full poloidal flux (Wb) instead of flux per radian (Wb/rad).
Psi values scale by 2π; per-psi derivatives scale by 1/(2π).

In [7]:
eq11 = eq.cocosify(11, copy=True)

print(f'psi_bdy (COCOS 1):  {eq.psi_boundary:.6f} Wb/rad')
print(f'psi_bdy (COCOS 11): {eq11.psi_boundary:.6f} Wb')
print(f'Ratio: {eq11.psi_boundary / eq.psi_boundary:.4f}  (2π = {2*np.pi:.4f})')
print()
print(f"pprime[0] (COCOS 1):  {eq.pprime[0]:.4e}")
print(f"pprime[0] (COCOS 11): {eq11.pprime[0]:.4e}")
print(f'Ratio: {eq11.pprime[0] / eq.pprime[0]:.4f}  (1/2π = {1/(2*np.pi):.4f})')

psi_bdy (COCOS 1):  0.044025 Wb/rad
psi_bdy (COCOS 11): 0.276616 Wb
Ratio: 6.2832  (2π = 6.2832)

pprime[0] (COCOS 1):  3.3203e+05
pprime[0] (COCOS 11): 5.2844e+04
Ratio: 0.1592  (1/2π = 0.1592)


## 4. Flip Bt and Ip directions

Negates toroidal field (BCENTR, FPOL) and plasma current (CURRENT),
plus psi fields and their derivatives.

In [8]:
eq_flip = eq.flip_Bt_Ip(copy=True)

print('--- After flip_Bt_Ip ---')
print(f'Ip:      {eq.Ip:>12,.0f}  →  {eq_flip.Ip:>12,.0f}')
print(f'B0:      {eq.B_center:>12.3f}  →  {eq_flip.B_center:>12.3f}')
print(f'psi_bdy: {eq.psi_boundary:>12.6f}  →  {eq_flip.psi_boundary:>12.6f}')
print(f'FPOL[0]: {eq.fpol[0]:>12.4f}  →  {eq_flip.fpol[0]:>12.4f}')
print(f'q(0):    {eq.qpsi[0]:>12.4f}  →  {eq_flip.qpsi[0]:>12.4f}  (unchanged)')

--- After flip_Bt_Ip ---
Ip:         1,200,040  →    -1,200,040
B0:             2.024  →        -2.024
psi_bdy:     0.044025  →     -0.044025
FPOL[0]:       3.4923  →       -3.4923
q(0):          1.2095  →        1.2095  (unchanged)


## 5. Full OMFIT-equivalent workflow

Load as COCOS 7, convert to COCOS 1, then flip Bt/Ip — all in place.

In [9]:
new_eq = read_geqdsk(gfile_path, cocos=7)
new_eq.cocosify(1)   # in-place
new_eq.flip_Bt_Ip()  # in-place

print(f'COCOS: {new_eq.cocos}')
print(f'Ip:    {new_eq.Ip:,.0f} A')
print(f'B0:    {new_eq.B_center:.3f} T')

COCOS: 1
Ip:    -1,200,040 A
B0:    -2.024 T


## 6. Save to disk

In [10]:
out_path = os.path.join(tempfile.gettempdir(), 'modified.geqdsk')
new_eq.save(out_path)
print(f'Saved to: {out_path}')

# Verify round-trip
check = read_geqdsk(out_path, cocos=1)
print(f'Re-read: Ip={check.Ip:,.0f} A, B0={check.B_center:.3f} T')
print(f'psi_RZ matches: {np.allclose(check.psi_RZ, new_eq.psi_RZ, rtol=1e-7)}')

Saved to: /var/folders/wz/x_qd5n8d62s0qj0_kcyyqf780000gn/T/modified.geqdsk
Re-read: Ip=-1,200,040 A, B0=-2.024 T
psi_RZ matches: True


## 7. Save to / load from HDF5

`to_bytes()` serialises to an ASCII byte string suitable for
opaque HDF5 storage (matching `bouquet.utils.store_equilibrium`).

In [11]:
import h5py

h5_path = os.path.join(tempfile.gettempdir(), 'equilibria_demo.h5')

# Store
raw_bytes = new_eq.to_bytes()
with h5py.File(h5_path, 'w') as hf:
    hf.create_dataset('equilibrium', data=np.void(raw_bytes))
    hf['equilibrium'].attrs['cocos'] = new_eq.cocos
print(f'Stored {len(raw_bytes):,} bytes in HDF5')

# Retrieve
with h5py.File(h5_path, 'r') as hf:
    stored_bytes = bytes(hf['equilibrium'][()])
    stored_cocos = int(hf['equilibrium'].attrs['cocos'])

eq_h5 = GEQDSKEquilibrium.from_bytes(stored_bytes, cocos=stored_cocos)
print(f'Loaded: COCOS={eq_h5.cocos}, Ip={eq_h5.Ip:,.0f} A, B0={eq_h5.B_center:.3f} T')
print(f'psi_RZ matches: {np.allclose(eq_h5.psi_RZ, new_eq.psi_RZ)}')

# Cleanup
for p in [out_path, h5_path]:
    if os.path.exists(p):
        os.remove(p)
print('Done.')

Stored 1,112,663 bytes in HDF5
Loaded: COCOS=1, Ip=-1,200,040 A, B0=-2.024 T
psi_RZ matches: True
Done.
